In [1]:
import numpy as np

def sigmoid(x): # 시그모이드 함수
    return 1. / (1. + np.exp(-x))

In [2]:
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [3]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.xdata = xdata.reshape(4,2) # 입력층 데이터
        self.tdata = tdata.reshape(4,1) # 정답 데이터

        self.W2 = np.random.rand(2,4) # 은닉층 가중치
        self.b2 = np.random.rand(4) # 은닉층 바이어스

        self.W3 = np.random.rand(4,2) # 은닉층
        self.b3 = np.random.rand(2) # 은닉층

        self.W4 = np.random.rand(2,1)
        self.b4 = np.random.rand(1)

        self.learning_rate = 1e-2 # 학습률

    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)
        z4 = np.dot(a3, self.W4) + self.b4
        y = a4 = sigmoid(z4)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1-y)+delta))

    def train(self, epoch=10000):
        f = lambda x : self.loss_val()
        print("Initial loss value = ", self.loss_val())
        for step in range(epoch+1):
            self.W2 -= self.learning_rate * numerical_derivative(f,self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f,self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f,self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f,self.b3)
            self.W4 -= self.learning_rate * numerical_derivative(f,self.W4)
            self.b4 -= self.learning_rate * numerical_derivative(f,self.b4)
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ",self.loss_val()) # 학습중 손실값 출력
    
    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2,self.W3) + self.b3
        a3 = sigmoid(z3)
        z4 = np.dot(a3, self.W4) + self.b4
        y = a4 = sigmoid(z4)
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    def accuracy(self, test_xdata, test_tdata):
        matched_list = [] # 정답과 일치한 내용 저장
        not_matched_list = [] # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index) # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index) # 정답과 일치한 내용 추가

        accuracy_val = len(matched_list) / len(test_xdata) # 정확도 계산
        return accuracy_val

In [6]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train(epoch=30000)

Initial loss value =  5.098258926095328
step =  0 loss value =  5.026769031360885
step =  1000 loss value =  2.7724024881008162
step =  2000 loss value =  2.772296019968646
step =  3000 loss value =  2.7721852686316244
step =  4000 loss value =  2.772068442124903
step =  5000 loss value =  2.771943534236945
step =  6000 loss value =  2.771808244461566
step =  7000 loss value =  2.7716598774412864
step =  8000 loss value =  2.771495213182483
step =  9000 loss value =  2.7713103355927564
step =  10000 loss value =  2.771100401119427
step =  11000 loss value =  2.770859320195833
step =  12000 loss value =  2.7705793096579185
step =  13000 loss value =  2.7702502504355433
step =  14000 loss value =  2.7698587446599765
step =  15000 loss value =  2.7693866967366247
step =  16000 loss value =  2.7688091184120274
step =  17000 loss value =  2.7680906269756202
step =  18000 loss value =  2.767179660293848
step =  19000 loss value =  2.765998534258368
step =  20000 loss value =  2.7644255666225

In [7]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  1
------------------
Accuracy =>  0.75


In [8]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train(epoch=50000)

Initial loss value =  3.723004833405862
step =  0 loss value =  3.683544796992557
step =  1000 loss value =  2.7731863194755833
step =  2000 loss value =  2.773083712049278
step =  3000 loss value =  2.7729851640812577
step =  4000 loss value =  2.772889812901526
step =  5000 loss value =  2.772796855973511
step =  6000 loss value =  2.7727055346039315
step =  7000 loss value =  2.7726151192580915
step =  8000 loss value =  2.7725248959065345
step =  9000 loss value =  2.772434152892301
step =  10000 loss value =  2.7723421678385596
step =  11000 loss value =  2.772248194116805
step =  12000 loss value =  2.7721514463660704
step =  13000 loss value =  2.7720510844907187
step =  14000 loss value =  2.7719461954622036
step =  15000 loss value =  2.7718357720976208
step =  16000 loss value =  2.7717186877694706
step =  17000 loss value =  2.771593665691185
step =  18000 loss value =  2.771459240985495
step =  19000 loss value =  2.7713137131215575
step =  20000 loss value =  2.77115508541

In [9]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
------------------
Accuracy =>  1.0


In [10]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.xdata = xdata.reshape(4,2) # 입력층 데이터
        self.tdata = tdata.reshape(4,1) # 정답 데이터

        self.W2 = np.random.rand(2,4) # 은닉층 가중치
        self.b2 = np.random.rand(4) # 은닉층 바이어스

        self.W3 = np.random.rand(4,2) # 은닉층
        self.b3 = np.random.rand(2) # 은닉층

        self.W4 = np.random.rand(2,1)
        self.b4 = np.random.rand(1)

        self.learning_rate = 1e-1 # 학습률

    def loss_val(self):
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)
        z4 = np.dot(a3, self.W4) + self.b4
        y = a4 = sigmoid(z4)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1-y)+delta))

    def train(self, epoch=10000):
        f = lambda x : self.loss_val()
        print("Initial loss value = ", self.loss_val())
        for step in range(epoch+1):
            self.W2 -= self.learning_rate * numerical_derivative(f,self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f,self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f,self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f,self.b3)
            self.W4 -= self.learning_rate * numerical_derivative(f,self.W4)
            self.b4 -= self.learning_rate * numerical_derivative(f,self.b4)
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ",self.loss_val()) # 학습중 손실값 출력
    
    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2,self.W3) + self.b3
        a3 = sigmoid(z3)
        z4 = np.dot(a3, self.W4) + self.b4
        y = a4 = sigmoid(z4)
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    def accuracy(self, test_xdata, test_tdata):
        matched_list = [] # 정답과 일치한 내용 저장
        not_matched_list = [] # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index) # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index) # 정답과 일치한 내용 추가

        accuracy_val = len(matched_list) / len(test_xdata) # 정확도 계산
        return accuracy_val

In [11]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train(epoch=20000)

Initial loss value =  3.606029953682074
step =  0 loss value =  3.2895614295891127
step =  1000 loss value =  2.7715128918978604
step =  2000 loss value =  2.7678953610628856
step =  3000 loss value =  2.6761670679699403
step =  4000 loss value =  0.12561994548228872
step =  5000 loss value =  0.032676757427607686
step =  6000 loss value =  0.018323004352591397
step =  7000 loss value =  0.012610220614022973
step =  8000 loss value =  0.009552719768118171
step =  9000 loss value =  0.007652692095616805
step =  10000 loss value =  0.006360156188283457
step =  11000 loss value =  0.005425789934911445
step =  12000 loss value =  0.004720152489857563
step =  13000 loss value =  0.004169341801830512
step =  14000 loss value =  0.0037280904974852135
step =  15000 loss value =  0.0033671287495124335
step =  16000 loss value =  0.003066700176424159
step =  17000 loss value =  0.00281299551885077
step =  18000 loss value =  0.0025960781267385887
step =  19000 loss value =  0.0024086203336421592

In [12]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
------------------
Accuracy =>  1.0


In [13]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train()

Initial loss value =  3.650361883628812
step =  0 loss value =  3.3457078559907223
step =  1000 loss value =  2.7657820562812585
step =  2000 loss value =  2.281758915464237
step =  3000 loss value =  0.056742496762344115
step =  4000 loss value =  0.01837622312366726
step =  5000 loss value =  0.010576845531943819
step =  6000 loss value =  0.007336872256438686
step =  7000 loss value =  0.005584175536642321
step =  8000 loss value =  0.0044925896496186115
step =  9000 loss value =  0.0037501255581280607
step =  10000 loss value =  0.0032136638882357405


In [14]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
------------------
Accuracy =>  1.0


In [15]:
xdata = np.array([[0,0], [0,1], [1,0], [1,1]]) # 학습 데이터
tdata = np.array([0,1,1,0]) # 정답 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)
XOR_obj.train(epoch=5000)

Initial loss value =  4.364244939665479
step =  0 loss value =  3.8248537957599344
step =  1000 loss value =  2.7723015934798516
step =  2000 loss value =  2.7703688397592066
step =  3000 loss value =  2.759646008271948
step =  4000 loss value =  2.0998309335738945
step =  5000 loss value =  0.5834078430071752


In [16]:
test_data = np.array([[0,0],[0,1],[1,0],[1,1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, " = ", logical_val)

print('------------------')
test_tdata = np.array([0,1,1,0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print("Accuracy => ", accuracy_ret)

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0
------------------
Accuracy =>  1.0
